In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

In [ ]:
MODEL_PATH = Path('./artifacts/3_glioma_best_model.joblib')
INPUT_PATH = Path('./data/preprocessed/3_glioma_grading_clinical_and_mutation_features_preprocessed_df.csv')
OUTPUT_PATH = Path('./data/preprocessed/3_glioma_inference_predictions.csv')
LABEL = 'Grade'

assert MODEL_PATH.exists(), f'Model not found: {MODEL_PATH.resolve()}'
assert INPUT_PATH.exists(), f'Input file not found: {INPUT_PATH.resolve()}'

bundle = joblib.load(MODEL_PATH)
if isinstance(bundle, dict):
    model = bundle.get('model', bundle.get('best_model', None))
    feature_columns = bundle.get('feature_columns', None)
else:
    model = bundle
    feature_columns = None

if model is None:
    raise ValueError('No model object found in artifact.')

In [ ]:
df = pd.read_csv(INPUT_PATH)
X = df.drop(columns=[LABEL], errors='ignore').copy()

if feature_columns is not None:
    missing = [c for c in feature_columns if c not in X.columns]
    if missing:
        raise ValueError(f'Missing required feature columns: {missing[:10]}')
    X = X.reindex(columns=feature_columns)

pred = model.predict(X)

out = pd.DataFrame({'prediction': pred})
if hasattr(model, 'predict_proba'):
    proba = model.predict_proba(X)
    if proba.shape[1] == 2:
        out['prob_grade_1'] = proba[:, 1]

if LABEL in df.columns:
    out.insert(0, 'true_label', df[LABEL].values)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_PATH, index=False)
print(f'✅ Saved predictions to: {OUTPUT_PATH.resolve()}')
out.head()